# ਪਾਠ 11 - ਏਜੰਟ ਤੋਂ ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੌਲ


## ਸੈੱਟਅਪ


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity python-dotenv

In [ ]:
import os
import dotenv
from agent_framework import tool, AgentResponseUpdate, WorkflowBuilder
from agent_framework.foundry import FoundryChatClient
from azure.identity import DefaultAzureCredential

dotenv.load_dotenv()

endpoint = os.getenv("AZURE_AI_PROJECT_ENDPOINT")
deployment_name = os.getenv("AZURE_AI_MODEL_DEPLOYMENT_NAME")

missing = [k for k, v in {
    "AZURE_AI_PROJECT_ENDPOINT": endpoint,
    "AZURE_AI_MODEL_DEPLOYMENT_NAME": deployment_name
}.items() if not v]

if missing:
    raise ValueError(
        f"Missing required environment variables: {', '.join(missing)}. "
        "Please set them as environment variables (e.g., in your .env file or shell environment)."
    )

In [ ]:
# Create the Microsoft Foundry client
client = FoundryChatClient(
    project_endpoint=endpoint,
    model=deployment_name,
    credential=DefaultAzureCredential()
)

## A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ?

**ਏਜੰਟ-ਟੂ-ਏਜੰਟ (A2A) ਪ੍ਰੋਟੋਕੋਲ** ਇੱਕ ਖੁੱਲ੍ਹਾ ਮਿਆਰ ਹੈ ਜੋ ਏਆਈ ਏਜੰਟਾਂ ਨੂੰ ਕੰਮ ਕਰਨਾ ਸਵਾਲ ਕਰਦਾ ਹੈ,
ਇਕ ਦੂਜੇ ਨੂੰ ਲੱਭਣ ਅਤੇ ਸਹਿਯੋਗ ਕਰਨ ਵਿਚ ਬਦਨਾਮ, ਭਾਵੇਂ ਉਹ ਵੱਖ-ਵੱਖ ਫਰੇਮਵਰਕਾਂ 'ਤੇ ਬਣੇ ਹੋਣ ਜਾਂ
ਵੱਖ-ਵੱਖ ਸੇਵਾਵਾਂ ਵੱਲੋਂ ਹੋਸਟ ਕੀਤੇ ਗਏ ਹੋਣ.

ਮੁੱਖ ਧਾਰਣਾ:

- **ਖੋਜ** – ਏਜੰਟ ਇੱਕ *ਏਜੰਟ ਕਾਰਡ* ਪ੍ਰਕਾਸ਼ਿਤ ਕਰਦੇ ਹਨ ਜੋ ਉਹਨਾਂ ਦੀਆਂ ਯੋਗਤਾਵਾਂ ਦਾ ਵੇਰਵਾ ਦਿੰਦਾ ਹੈ, ਜਿਸ ਨਾਲ
  ਹੋਰ ਏਜੰਟਾਂ (ਜਾਂ ਆਰਕੀਸਟਰੇਟਰਾਂ) ਲਈ ਇੱਕ ਕੰਮ ਲਈ ਸਹੀ ਵਿਸ਼ੇਸ਼ਗਿਆ ਨੂੰ ਲੱਭਣਾ ਆਸਾਨ ਹੁੰਦਾ ਹੈ.
- **ਸੁਨੇਹਾ ਭੇਜਣਾ** – ਏਜੰਟ ਇੱਕ ਆਮ ਪ੍ਰੋਟੋਕੋਲ ਰਾਹੀਂ ਸੰਰਚਿਤ ਸੁਨੇਹੇ ਬਦਲਦੇ ਹਨ, ਤਾਂ ਜੋ ਇੱਕ
  ਇੱਕ ਏਜੰਟ ਵੱਲੋਂ ਕੀਤੀ ਗਈ ਬੇਨਤੀ ਦੂਜੇ ਦੁਆਰਾ ਉਸ ਦੀ ਅੰਦਰੂਨੀ ਕਾਰਜਵਾਈ ਤੋਂ ਬਿਨਾਂ ਸਮਝੀ ਅਤੇ ਪੂਰੀ ਕੀਤੀ ਜਾ ਸਕੇ.
  ਲਾਗੂ ਕਰਨ ਦੇ ਤਰੀਕੇ ਨੂੰ ਸਮਝੇ ਬਿਨਾਂ.
- **ਕੰਮ ਦਾ ਜੀਵਨਚਕਰ** – A2A ਹਾਲਤਾਂ ਨੂੰ ਪਰਿਭਾਸ਼ਿਤ ਕਰਦਾ ਹੈ ਜਿਵੇਂ ਕਿ *ਸਮਰਪਿਤ*, *ਕੰਮ ਕਰ ਰਿਹਾ*, *ਪੂਰਾ ਕੀਤਾ* ਅਤੇ
  *ਫੇਲ*, ਆਰਕੀਸਟਰੇਟਰ ਨੂੰ ਪੂਰੀ ਦਿੱਖ ਦਿੰਦੇ ਹੋਏ ਕਿ ਦਿੱਤਾ ਗਇਆ ਕੰਮ ਕਿਵੇਂ ਅੱਗੇ ਵੱਧ ਰਿਹਾ ਹੈ.

ਇਸ ਪਾਠ ਵਿੱਚ ਅਸੀਂ A2A-ਸਟਾਈਲ ਸਹਿਯੋਗ ਦੀ ਨਕਲ ਕਰਦੇ ਹਾਂ ਜਿੱਥੇ ਤਿੰਨ ਵਿਸ਼ੇਸ਼ ਯਾਤਰਾ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜਿਆ ਜਾਂਦਾ ਹੈ,
ਜਿੱਥੇ ਹਰ ਏਜੰਟ ਆਪਣੀ ਮੁਹਾਰਤ ਦਿੰਦਾ ਹੈ ਅਤੇ ਨਤੀਜੇ ਅਗਲੇ ਨੂੰ ਦਿੰਦਾ ਹੈ.


## ਵਿਸ਼ੇਸ਼ਤ ਯਾਤਰਾ ਏਜੰਟ ਬਣਾਉਣਾ


In [ ]:
currency_agent = client.as_agent(
    name="CurrencyExchangeAgent",
    instructions="""You are a currency exchange specialist. You help travelers understand:
- Current exchange rates between currencies
- Best times to exchange money
- Tips for getting the best rates
When asked about a destination, provide relevant currency information.""",
)

activity_agent = client.as_agent(
    name="ActivityPlannerAgent",
    instructions="""You are a local activities specialist. You recommend:
- Must-see attractions and hidden gems
- Local experiences and cultural activities
- Restaurant and dining recommendations
Tailor suggestions to the traveler's interests.""",
)

travel_manager = client.as_agent(
    name="TravelManagerAgent",
    instructions="""You are a travel manager who coordinates between specialist agents.
When planning a trip:
1. Gather currency information from the currency specialist
2. Get activity recommendations from the activity planner
3. Synthesize everything into a cohesive travel brief
Present the final plan in an organized, easy-to-read format.""",
)

## ਵਰਕਫਲੋ ਰਾਹੀਂ ਬਹੁ-ਏਜੰਟ ਸਹਿਯੋਗ

ਅਸੀਂ ਤਿੰਨ ਏਜੰਟਾਂ ਨੂੰ ਇੱਕ ਲੜੀਵਾਰ ਵਰਕਫਲੋ ਵਿੱਚ ਜੋੜਦੇ ਹਾਂ ਜੋ A2A ਸੁਨੇਹਾ ਭੇਜਣ ਦੀ ਨਕਲ ਕਰਦਾ ਹੈ:

1. **CurrencyExchangeAgent** ਯੂਜ਼ਰ ਦੀ ਬੇਨਤੀ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਮੁਦਰਾ ਮਾਰਗਦਰਸ਼ਨ ਤਿਆਰ ਕਰਦਾ ਹੈ।
2. **ActivityPlannerAgent** ਸੰਵਰਧਿਤ ਸੰਦਰਭ ਪ੍ਰਾਪਤ ਕਰਦਾ ਹੈ ਅਤੇ ਸਰਗਰਮੀ ਦੀ ਸਿਫਾਰਿਸ਼ਾਂ ਜੋੜਦਾ ਹੈ।
3. **TravelManagerAgent** ਦੋਹਾਂ ਇਨਪੁਟਾਂ ਨੂੰ ਜੋੜ ਕੇ ਆਖਰੀ ਯਾਤਰਾ ਸੰਖੇਪ ਤਿਆਰ ਕਰਦਾ ਹੈ।


In [ ]:
workflow = WorkflowBuilder(start_executor=currency_agent) \
    .add_edge(currency_agent, activity_agent) \
    .add_edge(activity_agent, travel_manager) \
    .build()

last_author = None
events = workflow.run(
    "Plan a week-long trip to Tokyo. I love food, temples, and technology.",
    stream=True,
)
async for event in events:
    if event.type == "output" and isinstance(event.data, AgentResponseUpdate):
        update = event.data
        author = update.author_name
        if author != last_author:
            if last_author is not None:
                print()
            print(f"\n{'='*50}")
            print(f"🤖 {author}:")
            print(f"{'='*50}")
            last_author = author
        print(update.text, end="", flush=True)

## Understanding A2A in Production

In a production environment the A2A protocol unlocks powerful cross-service scenarios:

| Capability | Description |
|---|---|
| **Cross-framework interop** | An agent built with one framework can delegate tasks to an agent built with any other A2A-compliant framework, enabling true cross-organization interoperability. |
| **Service boundaries** | Agents can live in separate microservices, cloud regions, or even different organisations while still collaborating seamlessly. |
| **Dynamic discovery** | An orchestrator can query an Agent Card registry at runtime to find the best-suited specialist for a given sub-task. |
| **Streaming & push notifications** | A2A supports Server-Sent Events (SSE) for real-time progress updates and push notifications for long-running tasks. |

The workflow we built above is a simplified, in-process version of this pattern. In a real
deployment each agent would expose an HTTP endpoint, publish an Agent Card, and communicate
via the A2A JSON-RPC protocol.


## ਸਾਰ

ਇਸ ਪਾਠ ਵਿੱਚ ਤੁਸੀਂ ਸਿੱਖਿਆ:

1. **A2A ਪ੍ਰੋਟੋਕੋਲ ਕੀ ਹੈ** — ਏਜੰਟ-ਟੂ-ਏਜੰਟ ਖੋਜ, ਸੁਨੇਹਾ ਭੇਜਣ ਅਤੇ ਟਾਸਕ ਪ੍ਰਬੰਧਨ ਲਈ ਇੱਕ ਖੁਲਾ ਮਿਆਰ।
   ਅਤੇ ਟਾਸਕ ਪ੍ਰਬੰਧਨ।
2. **ਮਾਹਿਰ ਏਜੰਟ ਕਿਵੇਂ ਬਣਾਏ ਜਾਣ** — ਇੱਕ ਮੁਦਰਾ ਬਦਲਾਵ ਏਜੰਟ, ਇੱਕ ਕਾਰਜ ਯੋਜਕ ਏਜੰਟ,
   ਅਤੇ ਇੱਕ ਯਾਤਰਾ ਪ੍ਰਬੰਧਕ ਆਯੋਜਕ।
3. **ਏਜੰਟਸ ਨੂੰ ਵਰਕਫਲੋ ਵਿੱਚ ਕਿਵੇਂ ਜੋੜਨਾ ਹੈ** — ਏਜੰਟਾਂ ਵਿੱਚ ਲਗਾਤਾਰ ਸੁਨੇਹਾ ਪਾਸਿੰਗ ਮਾਡਲ ਕਰਨ ਲਈ `WorkflowBuilder` ਦੀ ਵਰਤੋਂ ਕਰਨਾ।
   ਸੁਨੇਹਾ ਪਾਸਿੰਗ ਮਾਡਲ ਕਰਨ ਲਈ `WorkflowBuilder` ਦੀ ਵਰਤੋਂ ਕਰਨਾ।
4. **A2A ਉਤਪਾਦਨ ਵਿੱਚ ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ** — ਗਤੀਸ਼ੀਲ ਖੋਜ ਅਤੇ ਸਟ੍ਰੀਮਿੰਗ ਅਪਡੇਟਾਂ ਨਾਲ ਕ੍ਰਾਸ-ਫ੍ਰੇਮਵਰਕ, ਕ੍ਰਾਸ-ਸੇਵਾ ਸਹਿਯੋਗ ਯੋਗ ਬਣਾਉਣਾ।
4. **A2A ਉਤਪਾਦਨ ਵਿੱਚ ਕਿਵੇਂ ਕੰਮ ਕਰਦਾ ਹੈ** — ਗਤੀਸ਼ੀਲ ਖੋਜ ਅਤੇ ਸਟ੍ਰੀਮਿੰਗ ਅਪਡੇਟਾਂ ਨਾਲ ਕ੍ਰਾਸ-ਫ੍ਰੇਮਵਰਕ, ਕ੍ਰਾਸ-ਸੇਵਾ ਸਹਿਯੋਗ ਯੋਗ ਬਣਾਉਣਾ।


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**ਅਸਵੀਕਾਰੋਪਣ**:
ਇਸ ਦਸਤਾਵੇਜ਼ ਦਾ ਅਨੁਵਾਦ ਏਆਈ ਅਨੁਵਾਦ ਸੇਵਾ [Co-op Translator](https://github.com/Azure/co-op-translator) ਦੀ ਵਰਤੋਂ ਕਰਕੇ ਕੀਤਾ ਗਿਆ ਹੈ। ਜਦੋਂ ਕਿ ਅਸੀਂ ਸਹੀਤਾਵਾਂ ਲਈ ਯਤਨਸ਼ੀਲ ਹਾਂ, ਕਿਰਪਾ ਕਰਕੇ ਧਿਆਨ ਰੱਖੋ ਕਿ ਸਵੈਚਾਲਿਤ ਅਨੁਵਾਦਾਂ ਵਿੱਚ ਗਲਤੀਆਂ ਜਾਂ ਅਸਮੱਤਿਆਵਾਂ ਹੋ ਸਕਦੀਆਂ ਹਨ। ਮੂਲ ਦਸਤਾਵੇਜ਼ ਆਪਣੀ ਮੂਲ ਭਾਸ਼ਾ ਵਿੱਚ ਅਧਿਕਾਰਕ ਸਰੋਤ ਮੰਨਿਆ ਜਾਣਾ ਚਾਹੀਦਾ ਹੈ। ਜਰੂਰੀ ਜਾਣਕਾਰੀ ਲਈ, ਪੇਸ਼ੇਵਰ ਮਨੁੱਖੀ ਅਨੁਵਾਦ ਦੀ ਸਿਫ਼ਾਰਸ਼ ਕੀਤੀ ਜਾਂਦੀ ਹੈ। ਅਸੀਂ ਇਸ ਅਨੁਵਾਦ ਦੇ ਉਪਯੋਗ ਤੋਂ ਪੈਦਾ ਹੋਣ ਵਾਲੀਆਂ ਕਿਸੇ ਵੀ ਗਲਤਫਹਿਮੀਆਂ ਜਾਂ ਗਲਤ ਵਿਆਖਿਆਵਾਂ ਲਈ ਜਵਾਬਦੇਹ ਨਹੀਂ ਹਾਂ।
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
